# Quantifying Mtb on a single-cell level across large tissue slices

First work on a single example to determine thresholds then iterate over all slices.

In [13]:
import os

import tensorflow as tf

# Set the log level to print ALL device placement information
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0'
tf.debugging.set_log_device_placement(True)

print("Device placement logging ENABLED. Run your prediction now.")

# After running this, execute your prediction cell:
# masks, details = model.predict_instances(dapi_channel[25000:28000, 25000:28000], n_tiles=PREDITION_TILES)

Device placement logging ENABLED. Run your prediction now.


In [10]:
model.predict_instances

<bound method StarDistBase._predict_instances_generator of StarDist2D(2D_versatile_fluo): YXC → YXC
├─ Directory: None
└─ Config2D(n_dim=2, axes='YXC', n_channel_in=1, n_channel_out=33, train_checkpoint='weights_best.h5', train_checkpoint_last='weights_last.h5', train_checkpoint_epoch='weights_now.h5', n_rays=32, grid=(2, 2), backbone='unet', n_classes=None, unet_n_depth=3, unet_kernel_size=[3, 3], unet_n_filter_base=32, unet_n_conv_per_depth=2, unet_pool=[2, 2], unet_activation='relu', unet_last_activation='relu', unet_batch_norm=False, unet_dropout=0.0, unet_prefix='', net_conv_after_unet=128, net_input_shape=[None, None, 1], net_mask_shape=[None, None, 1], train_shape_completion=False, train_completion_crop=32, train_patch_size=[256, 256], train_background_reg=0.0001, train_foreground_only=0.9, train_sample_cache=True, train_dist_loss='mae', train_loss_weights=[1, 0.2], train_class_weights=(1, 1), train_epochs=800, train_steps_per_epoch=400, train_learning_rate=0.0003, train_batch_s

In [14]:
%%time
# masks, _, _ = segmentation_model.eval(dapi_channel[25000:28000,25000:28000],                    
#                                diameter=40,
#                                min_size=0, progress=True
#                                )
masks, details = model.predict_instances(dapi_channel[25000:28000,25000:28000], 
                                          # n_tiles=PREDITION_TILES,
                                          )

base.py (406): Predicting on non-float input... ( forgot to normalize? )


CPU times: user 4min 32s, sys: 19.8 s, total: 4min 52s
Wall time: 22.9 s


In [8]:
viewer = napari.Viewer(title = 'testing stardist')
viewer.add_image(dapi_channel[25000:28000,25000:28000])
viewer.add_labels(masks)

<Labels layer 'masks' at 0x72391c82d790>

In [2]:
import glob
import os
from pathlib import Path

import dask.array as da
import napari
import numpy as np
import tifffile
import zarr
from csbdeep.utils import normalize
from stardist.models import StarDist2D
from tqdm.auto import tqdm

# 1. Load the StarDist model
# We select a pre-trained model versatile for fluorescence.
model = StarDist2D.from_pretrained('2D_versatile_fluo')


Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.


In [4]:
# Define the keywords for the preferred Zarr files
KEYWORDS = ('top', 'bot', 'left', 'right')

# The initial state: all Zarr file paths
all_zarr_paths = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*zarr')

# Group paths by their parent directory (mouse_N/zarr)
paths_by_parent = {}
for path in all_zarr_paths:
    parent_dir = os.path.dirname(path)
    if parent_dir not in paths_by_parent:
        paths_by_parent[parent_dir] = []
    paths_by_parent[parent_dir].append(path)

# Filter the paths
filtered_addresses = []
for paths in paths_by_parent.values():
    # Identify Zarr files that contain any of the keywords
    keyword_zarrs = [
        path for path in paths
        if any(keyword in os.path.basename(path) for keyword in KEYWORDS)
    ]

    # If keyword Zarrs exist, use only them; otherwise, use all paths in the group
    if keyword_zarrs:
        filtered_addresses.extend(keyword_zarrs)
    else:
        filtered_addresses.extend(paths)

# The streamlined result
print_list = [print(fn + '\n') for fn in filtered_addresses]

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_2/zarr/20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5573.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/zarr/rep2_mouse3_top.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/zarr/rep2_mouse3_bot.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_4/zarr/rep2_mouse4_bot.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_4/zarr/rep2_mouse4_top.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/zarr/rep2_mouse5_bot.zarr

/mnt/OPERA3/Nathan/data/macroh

In [6]:
NUCLEUS_DIAMETER = 40
PREDICTION_TILES = (16, 16)

In [5]:
for zarr_address in tqdm(filtered_addresses):
    # Find which, if any, keyword applies to this specific Zarr address
    split = next(
        (keyword for keyword in KEYWORDS if keyword in os.path.basename(zarr_address)),
        None  # Use None if no keyword is found
    )
    if split:
        # load s0 level of max projected images
        images = da.from_zarr(f"{zarr_address}/s0")
        # # --- load label ---
        # masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_{split}/0")   
    else:
        # original full image zarr, max projected
        images = da.from_zarr(f"{zarr_address}/0/0")[0].max(axis=1) 
        # # --- load label ---
        # masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_mtb/0")
    
    dapi_channel = images[0]
    break
    dapi_channel_norm = normalize(dapi_channel)
    
    print("Running StarDist prediction (this may take time)...")
    labels, details = model.predict_instances(dapi_channel_norm,
                                              n_tiles=PREDITION_TILES,
                                              show_progress=True
                                              )
        
    
    ## output labels to new zarr labels subdir under dapi_segmentation{_split}
    break
    print(zarr_address.split('whole_slice/')[1])
    print(images.shape)
    # print(masks.shape)
    print()

  0%|          | 0/45 [00:00<?, ?it/s]

In [29]:
model._guess_n_tiles(norm_seg_input)

(58, 81)

In [30]:
# Increase these numbers (e.g., (20, 20)) if you run out of memory.
prediction_tiles = (16, 16)
# prediction_tiles = model._guess_n_tiles(norm_seg_input)

In [ ]:
for zarr_address in tqdm(filtered_addresses):
    # Find which, if any, keyword applies to this specific Zarr address
    split = next(
        (keyword for keyword in KEYWORDS if keyword in os.path.basename(zarr_address)),
        None  # Use None if no keyword is found
    )
    # --- 1. Load Data Conditionally and Determine Output Path ---
    if split:
        # Load s0 level of max projected images for a SPLIT Zarr
        images = da.from_zarr(f"{zarr_address}/s0")
        # Set the output subdirectory name based on the split keyword
        output_sub_dir = f"dapi_segmentation_{split}"
        # masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_{split}/0")
    else:
        # Load max projected images for an ORIGINAL FULL IMAGE Zarr
        images = da.from_zarr(f"{zarr_address}/0/0")[0].max(axis=1)
        # Set the default output subdirectory name for whole images
        output_sub_dir = "dapi_segmentation"
        # masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_mtb/0")

    dapi_channel = images[0]
    dapi_channel_norm = normalize(dapi_channel)

    print(f"Running StarDist prediction for {os.path.basename(zarr_address)}...")
    labels, details = model.predict_instances(dapi_channel_norm,
                                              n_tiles=PREDITION_TILES,
                                              show_progress=True
                                              )

    # --- 2. Save Labels using NGFF-like Structure (Integrated) ---
    # Open the Zarr root for appending
    root = zarr.open_group(zarr_address, mode='a')
    labels_root = root.require_group("labels")
    
    # The label group will be created under 'labels' using the calculated name
    lbl_group = labels_root.require_group(output_sub_dir)
    
    # The group name for the segmentation labels
    dataset_name = "0"
    # Overwrite dataset '0' if it exists
    if dataset_name in lbl_group:
        del lbl_group[dataset_name]

    # Create and write the array
    # Using 'uint16' for efficient label storage
    arr_out = lbl_group.create_array(
        dataset_name,
        data=labels.astype(np.uint16),
        shape=labels.shape,
        dtype="uint16",
        chunks="auto",
        compressor=zarr.Blosc(cname='zstd', clevel=5, shuffle=zarr.Blosc.BITSHUFFLE)
    )

    # Add minimal NGFF label metadata
    lbl_group.attrs["multiscales"] = [{
        "name": output_sub_dir,
        "version": "0.4",
        "axes": [
            {"name": "y", "type": "space", "unit": "pixel"},
            {"name": "x", "type": "space", "unit": "pixel"},
        ],
        "datasets": [{"path": dataset_name}],
    }]
    lbl_group.attrs["image-label"] = {
        "version": "0.4",
        "source": "..",  # parent = image root
    }

    print(f"    ✓ wrote labels/{output_sub_dir}/0 [{labels.shape}]")

In [6]:
dapi_channel

dask.array<getitem, shape=(45850, 52070), dtype=>u2, chunksize=(1024, 1024), chunktype=numpy.ndarray>

In [28]:
dapi_channel_norm = normalize(dapi_channel)

In [29]:
model._guess_n_tiles(norm_seg_input)

(58, 81)

In [30]:
# Increase these numbers (e.g., (20, 20)) if you run out of memory.
prediction_tiles = (16, 16)
# prediction_tiles = model._guess_n_tiles(norm_seg_input)

In [31]:
# 4. Define output location
output_dir = "stardist_segmentation_test"
os.makedirs(output_dir, exist_ok=True)


In [34]:
print("Running StarDist prediction (this may take time)...")
labels, details = model.predict_instances(
    norm_seg_input,
    n_tiles=prediction_tiles,
    # show_progress=True
)
    
# 9. Save the resulting label mask
# We save each Z-plane as its own TIFF file.
output_filename = os.path.join(output_dir, "dapi_mask.tif")
print(f"Saving labels to {output_filename}...")
tifffile.imwrite(output_filename, labels,)# imagej=True)

Running StarDist prediction (this may take time)...


I0000 00:00:1762261879.760694   24334 service.cc:145] XLA service 0x778b7800a750 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1762261879.760771   24334 service.cc:153]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2025-11-04 13:11:19.806003: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-04 13:11:19.921475: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1762261880.933075   24334 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
100%|████████████████████████████████████████████████████████████████████████████████| 256/256 [02:27<00:00,  1.74it/s]


Saving labels to stardist_segmentation_test/dapi_mask.tif...


ValueError: the ImageJ format does not support data type 'i'

### Test the visualisation of a few

In [18]:
zarr_address = filtered_addresses[13]
print(zarr_address)

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_9/zarr/rep2_mouse9_top.zarr


In [19]:
os.listdir(zarr_address)

['.zattrs', '.zgroup', 'OME', 's0', 's1', 's2', 'labels']

In [ ]:
viewer = napari.Viewer()
viewer.add_image(dapi_channel)

image.py (274): data shape (45850, 52070) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.


In [5]:
viewer = napari.Viewer()
viewer.open(Path(zarr_address,'s0'), channel_axis=0)   # napari-ome-zarr plugin
# viewer.open(Path(zarr_address, 'labels', 'ground_truth_top'))


# # --- show in napari ---
# viewer = napari.Viewer()
# viewer.add_image(img, name="image")
# viewer.add_labels(lbl, name="ground_truth_mtb")
# napari.run()


ValueError: Given reader 'napari' is not a compatible reader for ['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.zarr/s0']. No compatible readers are available for ['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.zarr/s0'].

In [5]:

# --- load image (full-res level) ---
img = da.from_zarr(f"{zarr_address}/0/0")    # adjust path if your top-level differs

# --- load label ---
lbl = da.from_zarr(f"{zarr_address}/labels/ground_truth_mtb/0")


In [13]:
viewer = napari.Viewer(title = 'dapi segmentation')

In [7]:
viewer.add_labels(lbl)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Labels layer 'lbl' at 0x7e9cfc741dd0>

In [5]:
viewer.add_image(img)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f71830f9450>>
Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
Exception ignored in sys.unraisablehook: <built-in function unraisablehook>
Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/ipykernel/iostream.py", line 609, in flush
    if not evt.wait(self.flush_timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/threading.py", line 629, in wait
    signaled = self._cond.wait(timeout)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/threading.py", line 331, in wait
    gotit = waiter.acquire(True, timeout)
            ^^^^^

KeyboardInterrupt: 

In [6]:
img

dask.array<from-zarr, shape=(1, 3, 11, 41702, 58291), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>

# Segment

In [10]:
# 1. Load the StarDist model
# We select a pre-trained model versatile for fluorescence.
model = StarDist2D.from_pretrained('2D_versatile_fluo')


Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.


In [16]:
viewer.add_image(img[0,0,0,...])

scalar_field.py (197): data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.


<Image layer 'Image [1]' at 0x778b76c11810>

In [27]:
%%time
segmentation_input = img[0,0,...].max(axis=0).compute()

CPU times: user 3min 47s, sys: 1min 15s, total: 5min 3s
Wall time: 7min 20s


In [26]:
segmentation_input

dask.array<max-aggregate, shape=(41702, 58291), dtype=>u2, chunksize=(1024, 1024), chunktype=numpy.ndarray>

In [28]:
norm_seg_input = normalize(segmentation_input)

In [29]:
model._guess_n_tiles(norm_seg_input)

(58, 81)

In [30]:
# Increase these numbers (e.g., (20, 20)) if you run out of memory.
prediction_tiles = (16, 16)
# prediction_tiles = model._guess_n_tiles(norm_seg_input)

In [31]:
# 4. Define output location
output_dir = "stardist_segmentation_test"
os.makedirs(output_dir, exist_ok=True)


In [34]:
print("Running StarDist prediction (this may take time)...")
labels, details = model.predict_instances(
    norm_seg_input,
    n_tiles=prediction_tiles,
    # show_progress=True
)
    
# 9. Save the resulting label mask
# We save each Z-plane as its own TIFF file.
output_filename = os.path.join(output_dir, "dapi_mask.tif")
print(f"Saving labels to {output_filename}...")
tifffile.imwrite(output_filename, labels,)# imagej=True)

Running StarDist prediction (this may take time)...


I0000 00:00:1762261879.760694   24334 service.cc:145] XLA service 0x778b7800a750 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1762261879.760771   24334 service.cc:153]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2025-11-04 13:11:19.806003: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-04 13:11:19.921475: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1762261880.933075   24334 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
100%|████████████████████████████████████████████████████████████████████████████████| 256/256 [02:27<00:00,  1.74it/s]


Saving labels to stardist_segmentation_test/dapi_mask.tif...


ValueError: the ImageJ format does not support data type 'i'

In [37]:
viewer.add_labels(labels)

scalar_field.py (197): data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.


<Labels layer 'labels' at 0x778a082eee90>

In [39]:
import numpy as np

# Note: some older guides use ndi, but scipy.ndimage is standard
import scipy.ndimage
from skimage.segmentation import watershed

In [40]:
%%time

# 1. Create the topography for the watershed
# We calculate the distance from the background (labels == 0).
# This creates "mountains" inside the nuclei and "valleys" (0s)
# in the background.
distance_map = scipy.ndimage.distance_transform_edt(labels == 0)

# 2. Run the watershed algorithm
# This flows "downhill" from the 'labels' (markers) into the
# 'distance_map' valleys, filling the entire background.
# The result is a full Voronoi tessellation.
voronoi_cells = watershed(distance_map, labels)

# 3. Create the pseudo-cytoplasm mask
# We take the full Voronoi cells and set the original
# nuclear pixels (where labels > 0) back to 0.
pseudo_cytoplasm = voronoi_cells.copy()
pseudo_cytoplasm[labels > 0] = 0

# 'pseudo_cytoplasm' now contains the expanded regions,
# with the original nuclear areas cleared to 0.

ERROR! Session/line number was not unique in database. History logging moved to new session 3786
CPU times: user 1h 2min 59s, sys: 40.5 s, total: 1h 3min 39s
Wall time: 1h 3min 32s


In [51]:
np.save('voronoi_pseudo_cyto_seg.npy', pseudo_cytoplasm)

In [41]:
viewer.add_labels(pseudo_cytoplasm)

scalar_field.py (197): data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.


<Labels layer 'pseudo_cytoplasm' at 0x778ae934a350>

In [44]:
size_threshold = (pseudo_cytoplasm == 20977).sum()

In [49]:
viewer.add_labels((pseudo_cytoplasm == 20977).astype(bool))

scalar_field.py (197): data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.


<Labels layer 'Labels' at 0x778ae848ebd0>

In [46]:
size_threshold

18511

In [53]:
import numpy as np
import scipy.ndimage
from skimage.segmentation import watershed
from tqdm.auto import tqdm

In [ ]:
# 'labels' is your 2D numpy array of nuclear labels
# 'voronoi_cells' is the result of the full watershed expansion from the previous step
# 'distance_map' is the scipy.ndimage.distance_transform_edt(labels == 0)

# 1. Define your area limit
AREA_THRESHOLD = size_threshold #10000  # Example: 10,000 pixels

# 2. Initialize your final mask with the full Voronoi expansion
final_cells = voronoi_cells.copy()

# 3. Get the pixel counts (areas) for each fully-expanded cell
#    We use np.bincount(x.ravel()) as it is the fastest way.
areas = np.bincount(final_cells.ravel())

# 4. Find which label IDs have an area *larger* than the threshold
#    We ignore 0 (background)
label_ids_to_prune = np.where(areas > AREA_THRESHOLD)[0]
label_ids_to_prune = label_ids_to_prune[label_ids_to_prune > 0]

print(f"Found {len(label_ids_to_prune)} cells exceeding the {AREA_THRESHOLD} pixel threshold.")

# 5. Process only the cells that are too large
#    We will "prune" them by removing the pixels
#    that are farthest from *any* nucleus.
if len(label_ids_to_prune) > 0:
    
    # We need the distance from *any* nucleus.
    # This map gives a "distance" value to every pixel in the image.
    dist_from_any_nucleus = scipy.ndimage.distance_transform_edt(labels == 0)

    # 6. Loop over the oversized labels and prune them
    for label_id in tqdm(label_ids_to_prune, total = len(label_ids_to_prune)):
        
        # 7. Create a mask of *only* the pixels for this one oversized cell
        cell_mask = (final_cells == label_id)
        
        # 8. Find the pixel coordinates for this cell
        cell_coords = np.where(cell_mask)
        
        # 9. Get the distance for each pixel in that cell
        cell_pixel_distances = dist_from_any_nucleus[cell_coords]
        
        # 10. Sort the pixels by distance, from closest (0) to farthest
        sorted_indices = np.argsort(cell_pixel_distances)
        
        # 11. Find the indices of the pixels to *remove*
        #     We keep the first AREA_THRESHOLD pixels (the closest ones)
        #     and discard the rest.
        indices_to_remove = sorted_indices[AREA_THRESHOLD:]
        
        # 12. Get the coordinates of the pixels to remove
        coords_to_remove = (cell_coords[0][indices_to_remove], 
                            cell_coords[1][indices_to_remove])
        
        # 13. Set these pixels back to 0 (background)
        final_cells[coords_to_remove] = 0

print("Pruning complete.")

# 14. Create the final pseudo-cytoplasm mask
#     This mask contains only the expanded regions,
#     with the nuclei cleared to 0.
pseudo_cytoplasm = final_cells.copy()
pseudo_cytoplasm[labels > 0] = 0

Found 20718 cells exceeding the 18511 pixel threshold.


  0%|          | 0/20718 [00:00<?, ?it/s]

In [ ]:
viewer.add_labels(pseudo_cytoplasm, label='pruned')

## Quantify

2 versions, think the first is quicker

In [ ]:
zarr_addresses = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_*/zarr/2025*.zarr')
zarr_addresses = natsorted([fn for fn in zarr_addresses if 'notebook' not in fn])

results = []

for zarr_address in tqdm(zarr_addresses):
    try:
        # Load data
        images = da.from_zarr(f"{zarr_address}/0/0")
        masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_mtb/0")
        
        # Compute max projections lazily
        max_proj_ch1 = images[0, 1, :, :, :].max(axis=0)
        max_proj_ch2 = images[0, 2, :, :, :].max(axis=0)
        
        # Compute results
        max_proj_ch1_computed = max_proj_ch1.compute()
        max_proj_ch2_computed = max_proj_ch2.compute()
        masks_computed = masks.compute()
        masks_computed = label(masks_computed)
        # Use regionprops to efficiently compute measurements for all regions at once
        props = regionprops_table(masks_computed, 
                                 intensity_image=max_proj_ch1_computed,
                                 properties=['label', 'mean_intensity', 'area'])
        
        props_ch2 = regionprops_table(masks_computed,
                                     intensity_image=max_proj_ch2_computed,
                                     properties=['mean_intensity'])
        
        # Combine results
        for i, label_id in enumerate(props['label']):
            results.append({
                'zarr_address': os.path.basename(zarr_address),
                'region_id': label_id,
                'mean_intensity_ch1': props['mean_intensity'][i],
                'mean_intensity_ch2': props_ch2['mean_intensity'][i],
                'area': props['area'][i]
            })
    except:
        print(zarr_address, ' failed')
df_results = pd.DataFrame(results)

In [27]:
import os

import numpy as np
import pandas as pd
from natsort import natsorted
from skimage.measure import label, regionprops, regionprops_table
from tqdm.auto import tqdm

zarr_addresses = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_*/zarr/2025*.zarr')
zarr_addresses = natsorted([fn for fn in zarr_addresses if 'notebook' not in fn])

results = []

for zarr_address in tqdm(zarr_addresses):
    try:
        # Load data
        images = da.from_zarr(f"{zarr_address}/0/0")
        masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_mtb/0")
        
        # Compute masks once
        masks_computed = masks.compute()
        masks_computed = label(masks_computed)
        # Use regionprops to get bounding boxes efficiently
        regions = regionprops(masks_computed)
        
        region_results = []
        
        for region in tqdm(regions, desc="Processing regions", leave=False):
            region_id = region.label
            
            # Get bounding box with padding
            pad = 5
            rmin, cmin, rmax, cmax = region.bbox
            rmin = max(0, rmin - pad)
            rmax = min(masks_computed.shape[0], rmax + pad)
            cmin = max(0, cmin - pad)
            cmax = min(masks_computed.shape[1], cmax + pad)
            
            # Extract region mask
            region_mask_crop = masks_computed[rmin:rmax, cmin:cmax] == region_id
            
            # Extract image regions
            region_ch1 = images[0, 1, :, rmin:rmax, cmin:cmax]
            region_ch2 = images[0, 2, :, rmin:rmax, cmin:cmax]
            
            # Compute max projections for the small region
            max_proj_ch1_region = region_ch1.max(axis=0).compute()
            max_proj_ch2_region = region_ch2.max(axis=0).compute()
            
            # Calculate mean intensity
            mean_intensity_ch1 = np.mean(max_proj_ch1_region[region_mask_crop])
            mean_intensity_ch2 = np.mean(max_proj_ch2_region[region_mask_crop])
            
            region_results.append({
                'zarr_address': os.path.basename(zarr_address),
                'region_id': region_id,
                'mean_intensity_ch1': mean_intensity_ch1,
                'mean_intensity_ch2': mean_intensity_ch2,
                'area': region.area,
                'bbox_rmin': rmin,
                'bbox_rmax': rmax,
                'bbox_cmin': cmin,
                'bbox_cmax': cmax
            })
        
        results.extend(region_results)
        
    except Exception as e:
        print(f"{zarr_address} failed: {str(e)}")

df_results = pd.DataFrame(results)


  0%|          | 0/11 [00:00<?, ?it/s]

Processing regions:   0%|          | 0/16762 [00:00<?, ?it/s]

Processing regions:   0%|          | 0/1912 [00:00<?, ?it/s]

Processing regions:   0%|          | 0/396 [00:00<?, ?it/s]

Processing regions:   0%|          | 0/455 [00:00<?, ?it/s]

...

Processing regions:   0%|          | 0/109 [00:00<?, ?it/s]

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/zarr/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.zarr failed: No array found in store file:///mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/zarr/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.zarr/labels/ground_truth_mtb/0 at path 


Processing regions:   0%|          | 0/309 [00:00<?, ?it/s]

Processing regions:   0%|          | 0/6464 [00:00<?, ?it/s]

Processing regions:   0%|          | 0/243 [00:00<?, ?it/s]

In [28]:
df_results.to_pickle('/mnt/OPERA3/N')

,zarr_address,region_id,mean_intensity_ch1,mean_intensity_ch2,area,bbox_rmin,bbox_rmax,bbox_cmin,bbox_cmax
0,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,1,0.000000,0.000000,23.0,1267,1279,32515,32543
1,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,2,0.000000,0.000000,39.0,2019,2038,21217,21233
2,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,3,120.315068,110.698630,146.0,2433,2459,22886,22916
3,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,4,883.984615,250.476923,65.0,2496,2524,32883,32908
4,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,5,0.000000,0.000000,91.0,2500,2521,9392,9419
...,...,...,...,...,...,...,...,...,...
26999,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,239,137.860759,158.700422,237.0,36480,36503,21175,21213
27000,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,240,109.032258,114.903226,62.0,36519,36544,16639,16654
27001,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,241,302.518325,407.848168,191.0,36544,36574,20714,20741
27002,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,242,110.038760,117.930233,129.0,36584,36601,14265,14296


In [23]:
napari.Viewer().add_labels(masks_computed)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Labels layer 'masks_computed' at 0x79206607ae50>

In [18]:
max_proj_ch1

dask.array<max-aggregate, shape=(41702, 58291), dtype=>u2, chunksize=(1024, 1024), chunktype=numpy.ndarray>

In [9]:
images

dask.array<from-zarr, shape=(1, 3, 11, 41702, 58291), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>

In [10]:
masks

dask.array<from-zarr, shape=(41702, 58291), dtype=uint8, chunksize=(1304, 1822), chunktype=numpy.ndarray>

## Previous failsafe option

In [ ]:
zarr_addresses = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_*/zarr/2025*.zarr')
zarr_addresses = natsorted([fn for fn in zarr_addresses if 'notebook' not in fn])

results = []

for zarr_address in tqdm(zarr_addresses):
    try:
        # Load data
        images = da.from_zarr(f"{zarr_address}/0/0")
        masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_mtb/0")
        
        # Compute max projections lazily
        max_proj_ch1 = images[0, 1, :, :, :].max(axis=0)
        max_proj_ch2 = images[0, 2, :, :, :].max(axis=0)
        
        # Compute results
        max_proj_ch1_computed = max_proj_ch1.compute()
        max_proj_ch2_computed = max_proj_ch2.compute()
        masks_computed = masks.compute()
        
        # Use regionprops to efficiently compute measurements for all regions at once
        props = regionprops_table(masks_computed, 
                                 intensity_image=max_proj_ch1_computed,
                                 properties=['label', 'mean_intensity', 'area'])
        
        props_ch2 = regionprops_table(masks_computed,
                                     intensity_image=max_proj_ch2_computed,
                                     properties=['mean_intensity'])
        
        # Combine results
        for i, label_id in enumerate(props['label']):
            results.append({
                'zarr_address': os.path.basename(zarr_address),
                'region_id': label_id,
                'mean_intensity_ch1': props['mean_intensity'][i],
                'mean_intensity_ch2': props_ch2['mean_intensity'][i],
                'area': props['area'][i]
            })
    except:
        print(zarr_address, ' failed')
df_results = pd.DataFrame(results)